# SVG Generation from Text Prompts
DL Spring Kaggle Midterm\
Tanvir Shahriar\
ts5792

# Use of LLM Assistant
An LLM assistant (ChatGPT) was used as a development aid during this project, primarily for path handling, string manipulation, code style cleanup, sanity-check generation, and on-the-go debugging. It was also used for clarifying small implementation details, improving readability, and debugging during iterative development. All final code decisions, integration, and verification were completed by the myself.

In [14]:
import gc
import os
import re
import time
import random
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR_CANDIDATES = [
    Path('/kaggle/input/dl-spring-2026-svg-generation-from-text-prompts'),
    Path('/kaggle/input/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline'),
    Path('/kaggle/input'),
]
WORK_DIR = Path('/kaggle/working')
#MODEL_BASE_PATH = Path('/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1')
MODEL_CANDIDATES = [
    '/kaggle/input/qwen2-5-1-5b-instruct',
    'Qwen/Qwen2.5-1.5B-Instruct',
]

CONFIG = {
    'max_seq_length': 256,
    'max_train_samples': 2000,
    'max_eval_samples': 50,
    'lora_r': 4,
    'lora_alpha': 8,
    'learning_rate': 2e-4,
    'num_train_epochs': 1,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 32,
    'warmup_steps': 25,
    'weight_decay': 0.01,
    'logging_steps': 20,
    'save_steps': 0,
    'eval_steps': 0,
    'max_new_tokens': 128,
    'generation_batch_size': 1,
    'retrieval_top_k': 5,
    'output_dir': str(WORK_DIR / 'final_svg_lora_run'),
    'submission_path': str(WORK_DIR / 'submission.csv'),
}

SYSTEM_PROMPT = (
    'Generate only one compact valid SVG XML. '
    'Use width=256 height=256 viewBox="0 0 256 256". '
    'Keep under 16000 characters. '
    'Use only safe SVG tags. '
    'No markdown, no explanation, no script, no event handlers, no external references.'
)

def find_data_dir():
    seen = set()
    for root in DATA_DIR_CANDIDATES:
        if not root.exists():
            continue
        search_roots = [root]
        try:
            search_roots += [p for p in root.rglob('*') if p.is_dir()]
        except Exception:
            pass
        for candidate in search_roots:
            cand_str = str(candidate)
            if cand_str in seen:
                continue
            seen.add(cand_str)
            if (candidate / 'train.csv').exists() and (candidate / 'test.csv').exists() and (candidate / 'sample_submission.csv').exists():
                return candidate
    raise FileNotFoundError('Could not find train.csv, test.csv, and sample_submission.csv anywhere under /kaggle/input')

def looks_like_model_dir(path_obj):
    return (path_obj / 'config.json').exists() and ((path_obj / 'model.safetensors').exists() or (path_obj / 'pytorch_model.bin').exists() or (path_obj / 'model.safetensors.index.json').exists())

def find_model_name():
    for candidate in MODEL_CANDIDATES:
        try:
            p = Path(candidate)
            if p.exists() and looks_like_model_dir(p):
                return str(p)
            if p.exists():
                for sub in p.rglob('*'):
                    if sub.is_dir() and looks_like_model_dir(sub):
                        return str(sub)
        except Exception:
            pass
    input_root = Path('/kaggle/input')
    if input_root.exists():
        try:
            for sub in input_root.rglob('*'):
                 if sub.is_dir() and looks_like_model_dir(sub):
                     return str(sub)
        except Exception:
               pass
    return MODEL_CANDIDATES[-1]

def looks_like_model_dir(path_obj):
    return path_obj.is_dir() and (path_obj / 'config.json').exists() and ((path_obj / 'model.safetensors').exists() or (path_obj / 'pytorch_model.bin').exists() or (path_obj / 'model.safetensors.index.json').exists())

def resolve_model_path(base_path):
    if looks_like_model_dir(base_path):
        return str(base_path)
    if base_path.exists():
        for sub in base_path.rglob('*'):
            if looks_like_model_dir(sub):
                return str(sub)
    raise FileNotFoundError(f'Could not find a valid Transformers model directory under {base_path}')

DATA_DIR = find_data_dir()

MODEL_NAME = find_model_name()

#MODEL_NAME = resolve_model_path(MODEL_BASE_PATH)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Resolved DATA_DIR:', DATA_DIR)
print('Resolved MODEL_NAME:', MODEL_NAME)
CONFIG

Torch: 2.10.0+cu128
CUDA available: True
Resolved DATA_DIR: /kaggle/input/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline
Resolved MODEL_NAME: /kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1


{'max_seq_length': 256,
 'max_train_samples': 2000,
 'max_eval_samples': 50,
 'lora_r': 4,
 'lora_alpha': 8,
 'learning_rate': 0.0002,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 32,
 'warmup_steps': 25,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'save_steps': 0,
 'eval_steps': 0,
 'max_new_tokens': 128,
 'generation_batch_size': 1,
 'retrieval_top_k': 5,
 'output_dir': '/kaggle/working/final_svg_lora_run',
 'submission_path': '/kaggle/working/submission.csv'}

In [15]:
ALLOWED_TAGS = {
    'svg','g','path','rect','circle','ellipse','line','polyline','polygon','defs','use','symbol','clipPath','mask',
    'linearGradient','radialGradient','stop','text','tspan','title','desc','style','pattern','marker','filter'
}
MAX_SVG_LEN = 16000
MAX_PATHS = 256
SVG_RE = re.compile(r'<svg[\s\S]*?</svg>', flags=re.IGNORECASE)

def local_name(tag):
    return tag.split('}')[-1]

def normalize_space(text):
    text = '' if text is None else str(text)
    text = text.replace('\n', ' ').replace('\t', ' ')
    return re.sub(r'\s+', ' ', text).strip()

def normalize_prompt(text):
    text = normalize_space(text).lower().replace('grey', 'gray')
    text = re.sub(r'[^a-z0-9\s#.,:/+\-]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def enforce_svg_header(svg_text):
    if not isinstance(svg_text, str):
        return ''
    svg_text = svg_text.strip()
    start = svg_text.find('<svg')
    if start >= 0:
        svg_text = svg_text[start:]
    end = svg_text.rfind('</svg>')
    if end >= 0:
        svg_text = svg_text[:end + len('</svg>')]
    return svg_text.strip()

def simple_safe_svg():
    return (
        "<svg xmlns='http://www.w3.org/2000/svg' width='256' height='256' viewBox='0 0 256 256'>"
        "<rect x='0' y='0' width='256' height='256' fill='white'/>"
        "<circle cx='128' cy='128' r='52' fill='black'/>"
        "</svg>"
    )

def svg_is_valid(svg_text):
    if not isinstance(svg_text, str):
        return False
    svg_text = enforce_svg_header(svg_text)
    if len(svg_text) == 0 or len(svg_text) > MAX_SVG_LEN:
        return False
    trimmed = svg_text.lstrip()
    if not (trimmed.startswith('<svg') or trimmed.startswith('<ns0:svg') or '<svg' in trimmed[:40]):
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    if local_name(root.tag) != 'svg':
        return False
    # TS For training accept any parseable SVG viewBox/canvas.
    # TS: normalize generated outputs later.
    viewbox = root.attrib.get('viewBox')
    if viewbox is None and (root.attrib.get('width') is None or root.attrib.get('height') is None):
        return False
    path_count = 0
    for elem in root.iter():
        tag = local_name(elem.tag)
        if tag not in ALLOWED_TAGS:
            return False
        if tag == 'path':
            path_count += 1
            if path_count > MAX_PATHS:
                return False
        for attr_name, attr_val in elem.attrib.items():
            an = str(attr_name).lower()
            av = str(attr_val).lower()
            if an.startswith('on'):
                return False
            if 'javascript:' in av or 'http://' in av or 'https://' in av:
                return False
            if ('href' in an) and not str(attr_val).startswith('#') and '://' in av:
                return False
    return True

def sanitize_svg(svg_text):
    svg_text = enforce_svg_header(svg_text)
    if not svg_text or '<script' in svg_text.lower() or len(svg_text) > MAX_SVG_LEN:
        return None
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return None
    if local_name(root.tag) != 'svg':
        return None
    for elem in root.iter():
        if local_name(elem.tag) not in ALLOWED_TAGS:
            return None
        for attr_name, attr_val in list(elem.attrib.items()):
            an = str(attr_name).lower()
            av = str(attr_val).lower()
            if an.startswith('on') or 'javascript:' in av or 'http://' in av or 'https://' in av:
                return None
    root.set('xmlns', 'http://www.w3.org/2000/svg')
    root.set('width', '256')
    root.set('height', '256')
    root.set('viewBox', '0 0 256 256')
    cleaned = ET.tostring(root, encoding='unicode')
    if not svg_is_valid(cleaned):
        return None
    return cleaned

def validate_submission_svg(svg_text):
    issues = []

    if not isinstance(svg_text, str):
        issues.append('missing_svg_root_prefix')
    else:
        trimmed = str(svg_text).lstrip()
        if not (
            trimmed.startswith('<svg')
            or trimmed.startswith('<ns0:svg')
            or '<svg' in trimmed[:40]
        ):
            issues.append('missing_svg_root_prefix')

    if len(str(svg_text)) > MAX_SVG_LEN:
        issues.append('too_long')
    if '<script' in str(svg_text).lower():
        issues.append('script_tag')
    if re.search(r'on[a-zA-Z]+\s*=\s*', str(svg_text)):
        issues.append('event_handler')
    # TS: erros with parsing? check if parsing possible at all here?
    try:
        root = ET.fromstring(svg_text)
        if local_name(root.tag) != 'svg':
            issues.append('bad_root')
        if root.attrib.get('viewBox') != '0 0 256 256':
            issues.append('bad_viewBox')
        if sum(1 for el in root.iter() if local_name(el.tag) == 'path') > MAX_PATHS:
            issues.append('too_many_paths')
        for el in root.iter():
            if local_name(el.tag) not in ALLOWED_TAGS:
                issues.append('disallowed_tag')
                break
    except Exception:
        issues.append('xml_parse_error')
    return issues


In [16]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

# TS: insert debug here for quick checking.
DEBUG = False
DEBUG_TEST_ROWS = 50
if DEBUG:
    test_df = test_df.head(DEBUG_TEST_ROWS).copy()
    CONFIG['max_new_tokens'] = 128
    CONFIG['generation_batch_size'] = 2
    CONFIG['max_train_samples'] = min(CONFIG['max_train_samples'], 2000)
    CONFIG['max_eval_samples'] = min(CONFIG['max_eval_samples'], 100)

train_df['prompt_norm'] = train_df['prompt'].map(normalize_prompt)
test_df['prompt_norm'] = test_df['prompt'].map(normalize_prompt)

# TS: filter invalid svgs. can avoid copy()?
train_df = train_df[train_df['svg'].map(svg_is_valid)].copy()
train_df['svg_len'] = train_df['svg'].str.len()

train_df = (
    train_df.sort_values(['prompt_norm', 'svg_len'])
    .drop_duplicates('prompt_norm', keep='first')
    .reset_index(drop=True)
)

print('usable train rows:', len(train_df))
print('test rows:', len(test_df))
display(train_df[['prompt','svg_len']].head())


usable train rows: 45913
test rows: 1000


,prompt,svg_len
0,饼图是一个有三个扇区的圆圈，其中一个扇区填充了颜色。,456
1,"指纹扫描技术,数码设备上安全身份的确认方式",3590
2,"Σ, a symbol in the Greek alphabet, often used ...",222
3,3D Printer creating a glass.,8758
4,5-second refresh interval.,410


In [17]:
# TS: retrieval based on tf-idf similarity of prompts after norming, heuristics prefer more token-overlap and shorter svgs.
word_vec = TfidfVectorizer(analyzer='word', ngram_range=(1,3), min_df=2, max_df=0.98, strip_accents='unicode', sublinear_tf=True)
char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=2, max_df=0.995, sublinear_tf=True)
Xw = word_vec.fit_transform(train_df['prompt_norm'])
Xc = char_vec.fit_transform(train_df['prompt_norm'])
X_retr = sparse.hstack([0.8 * Xw, 1.2 * Xc], format='csr')

def token_overlap(a, b):
    sa, sb = set(a.split()), set(b.split())
    return 0.0 if not sa or not sb else len(sa & sb) / len(sa | sb)

def retrieve_svg(prompt_norm):
    qw = word_vec.transform([prompt_norm])
    qc = char_vec.transform([prompt_norm])
    qx = sparse.hstack([0.8 * qw, 1.2 * qc], format='csr')
    sims = linear_kernel(qx, X_retr)[0]
    top_k = min(CONFIG['retrieval_top_k'], len(train_df))
    cand_idx = np.argpartition(-sims, kth=top_k - 1)[:top_k]
    best_j, best_score = int(cand_idx[0]), -1e18
    for j in cand_idx:
        j = int(j)
        score = float(sims[j]) + 0.08 * token_overlap(prompt_norm, train_df.iloc[j]['prompt_norm']) - 0.000005 * len(train_df.iloc[j]['svg'])
        if score > best_score:
            best_score, best_j = score, j
    return train_df.iloc[best_j]['svg']


In [18]:
# TS: stratified sampling by length, keep diversity & avoid overfitting short svgs
def stratified_length_sample(df, n_rows, seed=SEED):
    if len(df) <= n_rows:
        return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    bins = pd.qcut(df['svg_len'], q=min(10, df['svg_len'].nunique()), duplicates='drop')
    df2 = df.assign(svg_bin=bins.astype(str))
    per_bin = max(1, n_rows // df2['svg_bin'].nunique())
    parts = []
    for _, chunk in df2.groupby('svg_bin', dropna=False):
        parts.append(chunk.sample(n=min(len(chunk), per_bin), random_state=seed))
    sampled = pd.concat(parts).drop(columns=['svg_bin'])
    if len(sampled) < n_rows:
        extra = df.drop(sampled.index, errors='ignore')
        need = min(n_rows - len(sampled), len(extra))
        if need > 0:
            sampled = pd.concat([sampled, extra.sample(n=need, random_state=seed)])
    return sampled.sample(frac=1.0, random_state=seed).head(n_rows).reset_index(drop=True)

sampled_df = stratified_length_sample(train_df, CONFIG['max_train_samples'])
eval_n = min(CONFIG['max_eval_samples'], max(20, len(sampled_df)//20))
eval_n = min(eval_n, max(1, len(sampled_df) - 50))
eval_idx = sampled_df.sample(n=eval_n, random_state=SEED).index
eval_df = sampled_df.loc[eval_idx].reset_index(drop=True)
train_fit_df = sampled_df.drop(index=eval_idx).reset_index(drop=True)

print('train rows:', len(train_fit_df))
print('eval rows:', len(eval_df))


train rows: 1950
eval rows: 50


In [19]:
# TS: quick check of retrieval quality,
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MODEL_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    device_map='auto' if torch.cuda.is_available() else None,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=0.0,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules = ['q_proj', 'v_proj']
)
model = get_peft_model(model, lora_cfg)

def make_chat_text(prompt, svg):
    return (
        f'System: {SYSTEM_PROMPT}\n'
        f'User: {normalize_space(prompt)}\n'
        f'Assistant: {svg}'
    )

train_ds = Dataset.from_pandas(train_fit_df[['prompt', 'svg']], preserve_index=False)
eval_ds = Dataset.from_pandas(eval_df[['prompt', 'svg']], preserve_index=False)
train_ds = train_ds.map(lambda x: {'text': make_chat_text(x['prompt'], x['svg'])})
eval_ds = eval_ds.map(lambda x: {'text': make_chat_text(x['prompt'], x['svg'])})

def tokenize_function(batch):
    toks = tokenizer(batch['text'], max_length=CONFIG['max_seq_length'], truncation=True, padding=False)
    toks['labels'] = toks['input_ids'].copy()
    return toks

train_tok = train_ds.map(tokenize_function, batched=True, remove_columns=train_ds.column_names)
eval_tok = eval_ds.map(tokenize_function, batched=True, remove_columns=eval_ds.column_names)

def collate_fn(features):
    max_len = max(len(f['input_ids']) for f in features)
    input_ids = []
    attention_mask = []
    labels = []
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    for f in features:
        ids = list(f['input_ids'])
        mask = list(f['attention_mask'])
        lab = list(f['labels'])
        pad_len = max_len - len(ids)
        input_ids.append([pad_id] * pad_len + ids)
        attention_mask.append([0] * pad_len + mask)
        labels.append([-100] * pad_len + lab)
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long),
    }

training_args = TrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    warmup_steps=CONFIG['warmup_steps'],
    logging_steps=CONFIG['logging_steps'],
    save_strategy='no',
    eval_strategy='no',
    fp16=True,
    optim='adamw_torch',
    report_to='none',
    gradient_checkpointing=True,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    data_collator=collate_fn,
)

# TS: final training here.
train_result = trainer.train()
trainer.save_model(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
train_result


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Map:   0%|          | 0/1950 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/1950 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Step,Training Loss
20,1.761650
40,1.502250
60,1.144058


TrainOutput(global_step=61, training_loss=1.4614819464136342, metrics={'train_runtime': 569.2953, 'train_samples_per_second': 3.425, 'train_steps_per_second': 0.107, 'total_flos': 3913826570459136.0, 'train_loss': 1.4614819464136342, 'epoch': 1.0})

In [24]:
import gc
del trainer
gc.collect()
torch.cuda.empty_cache()

# eval and generation with cache for speed.
model.eval()
model.config.use_cache = True

def extract_svg(text):
    m = SVG_RE.search(text)
    return m.group(0).strip() if m else None

def build_gen_text(prompt):
    return f'System: {SYSTEM_PROMPT}\nUser: {normalize_space(prompt)}\nAssistant: ' 

@torch.no_grad()
def generate_batch(prompts):
    texts = [build_gen_text(p) for p in prompts]
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=CONFIG['max_new_tokens'],
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = outputs[:, inputs['input_ids'].shape[1]:]
    decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return [extract_svg(x) for x in decoded]


In [21]:
rows = []
stats = {'generated_valid': 0, 'fallback_retrieval': 0, 'fallback_safe_svg': 0}
t0 = time.time()
bs = CONFIG['generation_batch_size']
total = len(test_df)


# TS: infeernce here.
for start in range(0, total, bs):
    batch = test_df.iloc[start:start+bs]
    raw_svgs = generate_batch(batch['prompt'].tolist())
    for row, raw_svg in zip(batch.itertuples(index=False), raw_svgs):
        cleaned = sanitize_svg(raw_svg) if raw_svg else None
        if cleaned is not None and svg_is_valid(cleaned):
            final_svg = cleaned
            stats['generated_valid'] += 1
        else:
            retr_svg = sanitize_svg(retrieve_svg(row.prompt_norm))
            if retr_svg is not None and svg_is_valid(retr_svg):
                final_svg = retr_svg
                stats['fallback_retrieval'] += 1
            else:
                final_svg = simple_safe_svg()
                stats['fallback_safe_svg'] += 1
        rows.append({'id': row.id, 'svg': final_svg})

    done = min(start + bs, total)
    elapsed = time.time() - t0
    rate = done / elapsed if elapsed > 0 else 0
    remaining = (total - done) / rate if rate > 0 else 0
    print(f'Done {done}/{total} | elapsed {elapsed/60:.1f} min | ETA {remaining/60:.1f} min')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

submission = pd.DataFrame(rows)
# TS: csv here.
submission.to_csv(CONFIG['submission_path'], index=False)
elapsed = (time.time() - t0) / 60
print('Saved submission:', CONFIG['submission_path'])
print('Stats:', stats)
print('Minutes:', round(elapsed, 2))
display(submission.head())


Done 1/1000 | elapsed 0.1 min | ETA 125.2 min
Done 2/1000 | elapsed 0.3 min | ETA 125.8 min
Done 3/1000 | elapsed 0.4 min | ETA 126.0 min
Done 4/1000 | elapsed 0.5 min | ETA 125.5 min
Done 5/1000 | elapsed 0.6 min | ETA 118.3 min
Done 6/1000 | elapsed 0.6 min | ETA 100.5 min
Done 7/1000 | elapsed 0.7 min | ETA 104.4 min
Done 8/1000 | elapsed 0.9 min | ETA 107.1 min
Done 9/1000 | elapsed 1.0 min | ETA 109.1 min
Done 10/1000 | elapsed 1.1 min | ETA 109.0 min
Done 11/1000 | elapsed 1.2 min | ETA 106.9 min
Done 12/1000 | elapsed 1.3 min | ETA 108.3 min
Done 13/1000 | elapsed 1.4 min | ETA 109.5 min
Done 14/1000 | elapsed 1.6 min | ETA 110.5 min
Done 15/1000 | elapsed 1.7 min | ETA 111.2 min
Done 16/1000 | elapsed 1.8 min | ETA 111.9 min
Done 17/1000 | elapsed 1.9 min | ETA 112.6 min
Done 18/1000 | elapsed 2.1 min | ETA 113.2 min
Done 19/1000 | elapsed 2.2 min | ETA 113.8 min
Done 20/1000 | elapsed 2.3 min | ETA 114.4 min
Done 21/1000 | elapsed 2.5 min | ETA 114.8 min
Done 22/1000 | elapsed

,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<ns0:svg xmlns:ns0=""http://www.w3.org/2000/svg..."
1,6eede943219547c22ac56085027d33cc,"<ns0:svg xmlns:ns0=""http://www.w3.org/2000/svg..."
2,ea045c7a247166f061ce504d9b7ccaab,"<ns0:svg xmlns:ns0=""http://www.w3.org/2000/svg..."
3,8fe82f3af89e487b31236ca829c3f071,"<ns0:svg xmlns:ns0=""http://www.w3.org/2000/svg..."
4,600464e4d92c75338462271a09b3f176,"<ns0:svg xmlns:ns0=""http://www.w3.org/2000/svg..."


In [22]:
# final checks.
validation_issues = submission['svg'].map(validate_submission_svg)
bad_rows = int((validation_issues.map(len) > 0).sum())
print('rows with validation issues:', bad_rows)
if bad_rows:
    display(submission.loc[validation_issues.map(len) > 0].assign(issues=validation_issues[validation_issues.map(len) > 0]).head())
else:
    print('All submission rows passed local validation checks.')


rows with validation issues: 0
All submission rows passed local validation checks.
